In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = (
    SparkSession.builder
    .appName("RapidKart Analytics")
    .config("spark.sql.shuffle.partitions", "8")  # laptop friendly
    .getOrCreate()
)

spark

In [3]:
from google.colab import files
files.upload()

Saving orders.csv to orders.csv
Saving users.csv to users.csv


{'orders.csv': b'order_id,user_id,order_time,order_value,delivery_time_minutes,payment_method,is_discount_used,city\r\n1,2,2024-02-04 21:07:00,374.96,9,UPI,True,Delhi\r\n2,3,2024-03-29 20:16:00,365.23,20,UPI,True,Pune\r\n3,5,2024-02-09 05:17:00,585.78,24,Card,True,Pune\r\n4,5,2024-02-12 20:28:00,544.33,18,UPI,False,Pune\r\n5,9,2024-01-23 18:29:00,756.63,22,Card,False,Delhi\r\n6,11,2024-02-15 09:22:00,712.96,20,UPI,False,Pune\r\n7,11,2024-02-19 19:32:00,708.44,16,Card,True,Pune\r\n8,11,2024-02-27 09:12:00,600.45,15,COD,False,Pune\r\n9,12,2024-03-04 20:32:00,508.56,16,UPI,False,Pune\r\n10,13,2024-01-14 08:22:00,693.54,13,Card,True,Pune\r\n11,15,2024-03-27 01:43:00,834.22,25,UPI,False,Delhi\r\n12,16,2024-02-18 19:07:00,857.84,10,COD,False,Mumbai\r\n13,16,2024-02-28 19:04:00,720.15,16,Card,False,Mumbai\r\n14,18,2024-02-22 08:59:00,628.4,12,COD,False,Delhi\r\n15,18,2024-03-04 10:23:00,303.08,19,UPI,True,Delhi\r\n16,19,2024-03-22 20:06:00,869.19,21,COD,False,Delhi\r\n17,20,2024-03-09 07:41:0

## Daily KPI Table
### How is the platform Performing?

In [4]:
from google.colab import drive
drive.mount('/content/drive')

events = spark.read.parquet(
"/content/drive/MyDrive/RapidKart/silver/events_sessionized"
)

users = spark.read.csv("users.csv", header=True, inferSchema=True)
orders = spark.read.csv("orders.csv", header=True, inferSchema=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
from pyspark.sql.functions import to_timestamp

orders = orders.withColumn(
    "order_time",
    to_timestamp("order_time")
)

In [10]:
events.createOrReplaceTempView("events")
orders.createOrReplaceTempView("orders")

In [15]:
daily_kpis = spark.sql("""
SELECT
    e.event_date,
    COUNT(DISTINCT e.user_id) AS dau,
    COUNT(DISTINCT e.derived_session_id) AS sessions,
    COUNT(DISTINCT o.order_id) AS orders,
    SUM(o.order_value) AS revenue,
    ROUND(AVG(o.delivery_time_minutes),2) AS avg_delivery_time
FROM events e
LEFT JOIN orders o
ON e.user_id = o.user_id
AND DATE(o.order_time) = e.event_date
GROUP BY e.event_date
ORDER BY e.event_date
""")

In [34]:
daily_kpis.show(10)

+----------+---+--------+------+------------------+-----------------+-------------------+
|event_date|dau|sessions|orders|           revenue|avg_delivery_time|    conversion_rate|
+----------+---+--------+------+------------------+-----------------+-------------------+
|2024-01-02| 15|      15|     1|           2897.46|             28.0|0.06666666666666667|
|2024-01-03| 26|      26|     7| 36182.50999999998|            15.48| 0.2692307692307692|
|2024-01-04| 47|      47|    11| 63325.48999999996|            16.57|0.23404255319148937|
|2024-01-05| 83|      83|    17|          85920.46|            16.81|0.20481927710843373|
|2024-01-06|100|     100|    17|          96316.65|            19.56|               0.17|
|2024-01-07|116|     117|    24|130013.69999999987|            19.54|0.20512820512820512|
|2024-01-08|187|     189|    35|185594.95999999985|            18.24|0.18518518518518517|
|2024-01-09|183|     183|    35| 204041.3699999998|            17.91| 0.1912568306010929|
|2024-01-1

In [17]:
from pyspark.sql.functions import col

daily_kpis = daily_kpis.withColumn(
    "conversion_rate",
    col("orders") / col("sessions")
)

In [18]:
daily_kpis.write.mode("overwrite").parquet(
"/content/drive/MyDrive/RapidKart/gold/daily_kpis"
)

## FUNNEL Analysis
### Where do users drop before ordering?


In [19]:
funnel = spark.sql("""
SELECT
    event_date,
    COUNT(DISTINCT CASE WHEN event_name='app_open' THEN user_id END) AS app_open,
    COUNT(DISTINCT CASE WHEN event_name='product_view' THEN user_id END) AS product_view,
    COUNT(DISTINCT CASE WHEN event_name='add_to_cart' THEN user_id END) AS add_to_cart,
    COUNT(DISTINCT CASE WHEN event_name='checkout_start' THEN user_id END) AS checkout,
    COUNT(DISTINCT CASE WHEN event_name='order_placed' THEN user_id END) AS orders
FROM events
GROUP BY event_date
ORDER BY event_date
""")

In [33]:
funnel.show(5)

+----------+--------+------------+-----------+--------+------+------------------+------------------+------------------+------------------+
|event_date|app_open|product_view|add_to_cart|checkout|orders|         view_rate|         cart_rate|     checkout_rate|     purchase_rate|
+----------+--------+------------+-----------+--------+------+------------------+------------------+------------------+------------------+
|2024-01-02|      15|          13|          7|       2|     1|0.8666666666666667|0.5384615384615384|0.2857142857142857|               0.5|
|2024-01-03|      25|          18|         15|      10|     7|              0.72|0.8333333333333334|0.6666666666666666|               0.7|
|2024-01-04|      47|          30|         23|      19|    11|0.6382978723404256|0.7666666666666667|0.8260869565217391|0.5789473684210527|
|2024-01-05|      83|          66|         48|      22|    17|0.7951807228915663|0.7272727272727273|0.4583333333333333|0.7727272727272727|
|2024-01-06|      99|      

In [21]:
funnel = funnel.withColumn(
    "view_rate",
    col("product_view") / col("app_open")
).withColumn(
    "cart_rate",
    col("add_to_cart") / col("product_view")
).withColumn(
    "checkout_rate",
    col("checkout") / col("add_to_cart")
).withColumn(
    "purchase_rate",
    col("orders") / col("checkout")
)

In [22]:
funnel.write.mode("overwrite").parquet(
"/content/drive/MyDrive/RapidKart/gold/funnel_metrics"
)

## Retention Cohorts
###

In [23]:
users = users.withColumn(
    "signup_date",
    to_timestamp("signup_date")
)

users.createOrReplaceTempView("users")

In [24]:
retention = spark.sql("""
SELECT
    u.user_id,
    DATE(u.signup_date) AS signup_date,
    DATE(e.event_time) AS activity_date
FROM users u
JOIN events e
ON u.user_id = e.user_id
""")

In [32]:
retention.show(10)

+-------+-----------+-------------+-----------+
|user_id|signup_date|activity_date|week_number|
+-------+-----------+-------------+-----------+
|      1| 2024-03-22|   2024-03-29|          1|
|      1| 2024-03-22|   2024-03-29|          1|
|      1| 2024-03-22|   2024-03-29|          1|
|      1| 2024-03-22|   2024-03-29|          1|
|      1| 2024-03-22|   2024-03-29|          1|
|      1| 2024-03-22|   2024-03-29|          1|
|      3| 2024-03-27|   2024-03-29|          0|
|      3| 2024-03-27|   2024-03-29|          0|
|      3| 2024-03-27|   2024-03-29|          0|
|      3| 2024-03-27|   2024-03-29|          0|
+-------+-----------+-------------+-----------+
only showing top 10 rows


In [27]:
from pyspark.sql.functions import datediff, floor

retention = retention.withColumn(
    "week_number",
    floor(datediff("activity_date","signup_date")/7)
)

In [28]:
cohort = retention.groupBy(
    "signup_date","week_number"
).count()

In [29]:
cohort.write.mode("overwrite").parquet(
"/content/drive/MyDrive/RapidKart/gold/retention_cohort"
)

## User Summary

In [36]:
user_summary = spark.sql("""
SELECT
    user_id,
    COUNT(DISTINCT derived_session_id) AS total_sessions,
    COUNT(CASE WHEN event_name='order_placed' THEN 1 END) AS total_orders
FROM events
GROUP BY user_id
ORDER BY user_id
""")

In [38]:
from pyspark.sql.functions import when, col

user_summary = user_summary.withColumn(
    "user_segment",
    when(col("total_orders") >= 10, "Power User")
    .when(col("total_orders") >= 3, "Regular User")
    .when(col("total_orders") >= 1, "Occasional User")
    .otherwise("Inactive")
)

In [39]:
user_summary.show(10)

+-------+--------------+------------+---------------+
|user_id|total_sessions|total_orders|   user_segment|
+-------+--------------+------------+---------------+
|      1|             1|           0|       Inactive|
|      2|             2|           1|Occasional User|
|      3|             1|           1|Occasional User|
|      4|             5|           0|       Inactive|
|      5|            10|           2|Occasional User|
|      6|             2|           0|       Inactive|
|      7|             7|           0|       Inactive|
|      8|             2|           0|       Inactive|
|      9|             5|           1|Occasional User|
|     10|             4|           0|       Inactive|
+-------+--------------+------------+---------------+
only showing top 10 rows


In [40]:
user_summary.write \
    .mode("overwrite") \
    .parquet("/content/drive/MyDrive/RapidKart/gold/user_summary")